# Approach 09  KD to Shallow Student

Train a student with fewer blocks to match the teacher's logits.

## Setup  install & versions

In [1]:
# Setup  install & versions
!pip install -q transformers datasets accelerate pandas matplotlib scipy

import torch
import transformers

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)

device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("device:", device)


zsh:1: command not found: pip


/Users/sumitjha/Downloads/nlp-layer-reduction/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch: 2.14.1
Transformers: 5.18.0
device: mps


## Reproducibility  seeds

In [2]:
# Reproducibility  seeds
import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


## Models

In [3]:
# Models
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_IDS = [
    "openai-community/gpt2",
    "HuggingFaceTB/SmolLM-135M",
]

tokenizers = {}
for mid in MODEL_IDS:
    tok = AutoTokenizer.from_pretrained(mid)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    tokenizers[mid] = tok


## Dataset

In [4]:
# Dataset
from datasets import load_dataset

dataset = load_dataset("Salesforce/wikitext", "wikitext-103-v1")

def join_nonempty(split):
    return "\n\n".join(x for x in split["text"] if x and x.strip())

train_text = join_nonempty(dataset["train"])[:300000]
validation_text = join_nonempty(dataset["validation"])[:100000]
test_text = join_nonempty(dataset["test"])[:100000]
print(dataset)


DatasetDict({
    test: Dataset({
        features: ['text'],
        num_rows: 4358
    })
    train: Dataset({
        features: ['text'],
        num_rows: 1801350
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 3760
    })
})


## Helpers  model architecture

In [5]:
# Helpers  model architecture
def get_layers(model):
    if hasattr(model, "transformer") and hasattr(model.transformer, "h"):
        return model.transformer.h
    return model.model.layers

def set_layers(model, blocks):
    blocks = torch.nn.ModuleList(blocks)
    if hasattr(model, "transformer") and hasattr(model.transformer, "h"):
        model.transformer.h = blocks
        model.config.n_layer = len(blocks)
    else:
        model.model.layers = blocks
        model.config.num_hidden_layers = len(blocks)
    return model

def load_fresh_model(model_id):
    model = AutoModelForCausalLM.from_pretrained(model_id)
    model.config.use_cache = False
    model.eval()
    return model

def hidden_size(model):
    cfg = model.config
    return getattr(cfg, "n_embd", None) or cfg.hidden_size


## Metrics  PPL / params / latency

In [6]:
# Metrics  PPL / params / latency
import math, time, gc

EVAL_BLOCK_SIZE = 256
EVAL_TOKENS = 3000

def calculate_ppl(model, text, model_id, block_size=EVAL_BLOCK_SIZE, max_tokens=EVAL_TOKENS):
    model.eval()
    tok = tokenizers[model_id]
    ids = tok(text, return_tensors="pt", add_special_tokens=False)["input_ids"][0]
    if max_tokens is not None:
        ids = ids[:max_tokens]
    n_blocks = len(ids) // block_size
    total_nll, total_tokens = 0.0, 0
    with torch.no_grad():
        for i in range(n_blocks):
            x = ids[i*block_size:(i+1)*block_size].unsqueeze(0).to(device)
            out = model(input_ids=x, labels=x, use_cache=False)
            c = x.numel() - 1
            total_nll += out.loss.item() * c
            total_tokens += c
    return math.exp(total_nll / total_tokens)

def parameter_count(model):
    return sum(p.numel() for p in model.parameters())

def measure_latency(model, text, model_id, seq_len=256, repeat=10):
    model.eval()
    tok = tokenizers[model_id]
    ids = tok(text, return_tensors="pt", add_special_tokens=False)["input_ids"][:, :seq_len].to(device)
    with torch.no_grad():
        for _ in range(3):
            _ = model(ids, use_cache=False)
        t0 = time.perf_counter()
        for _ in range(repeat):
            _ = model(ids, use_cache=False)
        t1 = time.perf_counter()
    return 1000.0 * (t1 - t0) / repeat

def free_model(m):
    del m
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


In [7]:
def get_layers(model):
    if hasattr(model, "transformer") and hasattr(model.transformer, "h"):
        return model.transformer.h
    return model.model.layers

def set_layers(model, blocks):
    blocks = torch.nn.ModuleList(blocks)
    if hasattr(model, "transformer") and hasattr(model.transformer, "h"):
        model.transformer.h = blocks
        model.config.n_layer = len(blocks)
    else:
        model.model.layers = blocks
        model.config.num_hidden_layers = len(blocks)
    return model

def load_fresh_model(model_id):
    model = AutoModelForCausalLM.from_pretrained(model_id)
    model.config.use_cache = False
    model.eval()
    return model

def hidden_size(model):
    cfg = model.config
    return getattr(cfg, "n_embd", None) or cfg.hidden_size


In [8]:
import math, time, gc

EVAL_BLOCK_SIZE = 256
EVAL_TOKENS = 3000

def calculate_ppl(model, text, model_id, block_size=EVAL_BLOCK_SIZE, max_tokens=EVAL_TOKENS):
    model.eval()
    tok = tokenizers[model_id]
    ids = tok(text, return_tensors="pt", add_special_tokens=False)["input_ids"][0]
    if max_tokens is not None:
        ids = ids[:max_tokens]
    n_blocks = len(ids) // block_size
    total_nll, total_tokens = 0.0, 0
    with torch.no_grad():
        for i in range(n_blocks):
            x = ids[i*block_size:(i+1)*block_size].unsqueeze(0).to(device)
            out = model(input_ids=x, labels=x, use_cache=False)
            c = x.numel() - 1
            total_nll += out.loss.item() * c
            total_tokens += c
    return math.exp(total_nll / total_tokens)

def parameter_count(model):
    return sum(p.numel() for p in model.parameters())

def measure_latency(model, text, model_id, seq_len=256, repeat=10):
    model.eval()
    tok = tokenizers[model_id]
    ids = tok(text, return_tensors="pt", add_special_tokens=False)["input_ids"][:, :seq_len].to(device)
    with torch.no_grad():
        for _ in range(3):
            _ = model(ids, use_cache=False)
        t0 = time.perf_counter()
        for _ in range(repeat):
            _ = model(ids, use_cache=False)
        t1 = time.perf_counter()
    return 1000.0 * (t1 - t0) / repeat

def free_model(m):
    del m
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


## Method  Student KD

In [9]:
# Method  Student KD

import torch.nn.functional as F

def make_student(model_id, keep_frac=0.5):
    m = load_fresh_model(model_id)
    blocks = list(get_layers(m))
    keep = max(1, int(len(blocks) * keep_frac))
    set_layers(m, blocks[:keep])
    return m

def train_student(student, teacher, model_id, steps=60, lr=2e-5, seq_len=256):
    tok = tokenizers[model_id]
    ids = tok(train_text[:20000], return_tensors="pt", add_special_tokens=False)["input_ids"][0]
    opt = torch.optim.AdamW(student.parameters(), lr=lr)
    for i in range(steps):
        start = (i * seq_len) % max(1, (len(ids) - seq_len))
        x = ids[start:start + seq_len].unsqueeze(0).to(device)
        s = student(input_ids=x).logits
        with torch.no_grad():
            t = teacher(input_ids=x).logits
        loss = F.kl_div(F.log_softmax(s / 2, -1), F.softmax(t / 2, -1),
                        log_target=False, reduction="batchmean") * 4
        opt.zero_grad(); loss.backward(); opt.step()
        if i % 50 == 0: print(i, round(loss.item(), 4))

rows = []
for mid in MODEL_IDS:
    teacher = load_fresh_model(mid).to(device)
    for keep_frac in [0.5, 0.7]:
        student = make_student(mid, keep_frac).to(device)
        student.train()
        pre = calculate_ppl(student, validation_text, mid)
        train_student(student, teacher, mid, steps=60)
        post = calculate_ppl(student, validation_text, mid)
        rows.append({"model": mid, "keep_frac": keep_frac, "ppl_before": pre, "ppl_after": post})
        print(mid, keep_frac, pre, "->", post)
        free_model(student)
    free_model(teacher)

import pandas as pd
pd.DataFrame(rows)


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 27820.42it/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 16295.83it/s]


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (22157 > 1024). Running this sequence through the model will result in indexing errors


...

Loading weights: 100%|██████████| 272/272 [00:00<00:00, 11863.10it/s]

0 2848.0


50 2448.0


HuggingFaceTB/SmolLM-135M 0.7 38515.31324211551 -> 32101.824636933707


,model,keep_frac,ppl_before,ppl_after
0,openai-community/gpt2,0.5,9478.145970,510.052793
1,openai-community/gpt2,0.7,3083.034984,241.025717
2,HuggingFaceTB/SmolLM-135M,0.5,162776.706439,147722.338093
3,HuggingFaceTB/SmolLM-135M,0.7,38515.313242,32101.824637


## Summary
Results are embedded above. See `approaches/reports/` for the matching report file (same numbering).